# SQL in Python - Connecting to and retrieving data from PostgreSQL

Previously, you have learned how to connect to a SQL database by using a SQL client such as DBeaver. Apart from connecting to databases, DBeaver also allows you to run SQL queries against the database, create new tables and populate them with data as well as retrieving the data.

Python also allows executing SQL queries and getting the result into a Python object, for example a Pandas data frame. Instead of exporting a .csv file from DBeaver you can directly get the data you need into Python and continue your work. In addition we can reduce the steps by connecting to the database from Python directly, eliminating the need for a separate SQL client.

After you have the data in Python in the required shape you can export the data into a .csv file. This file is for your own reference, please avoid sending .csv files around - database is the point of reference when it comes to data. 

Having a copy of a .csv file (or another format) can speed up your analysis work. Imagine that the query takes 25 minutes to run. If you made some mistakes in your Python code you might need to go back to the original dataset. Instead of having to rerun the SQL query and having to wait you can read in the .csv file you have previously saved on your hard disk into Python and continue with your analysis work. 

**In this notebook you will see 2 ways to connect to SQL-Databases and export the data to a CSV file**


## Creating a connection to a PostgreSQL database with Python

There are 2 python packages that are the "go-to" when it comes to connecting to SQL-Databases: `psycopg2` and `sqlalchemy` 

### Connecting via psycopg2

In [4]:
import pandas as pd
import psycopg2


In order to create a connection to our PostgreSQL database we need the following information:

- host = the address of the machine the database is hosted on
- port = the virtual gate number through which communication will be allowed
- database = the name of the database
- user = the name of the user
- password = the password of the user

Because we don't want that the database information is published on GitHub we put it into a `.env` file which is added into the `.gitignore`. 
In these kind of files you can store information that is not supposed to be published.
With the `dotenv` package you can read the `.env` files and get the variables.


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

The function from the psycopg2 package to create a connection is called `connect()`.
`connect()` expects the parameters listed above as input in order to connect to the database.

In [6]:
# Create connection object conn
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

OperationalError: connection to server on socket "/tmp/.s.PGSQL.5432" failed: FATAL:  database "jolenndarensbourg" does not exist


### Retrieving data from the database with psycopg2

Before we can use our connection to get data, we have to create a cursor. A cursor allows Python code to execute PostgreSQL commands in a database session.
A cursor has to be created with the `cursor()` method of our connection object conn.

In [ ]:
cur = conn.cursor()

Now we can run SQL-Queries with `cur.execute('QUERY')` and then run `cur.fetchall()` to get the data:

In [ ]:
cur.execute("SELECT * FROM eda.king_county_house_sales LIMIT 10")
cur.fetchall()

With `conn.close()` you can close the connection again.

In [ ]:
# close the connection
conn.close()

But we want to work with the data. The easiest way is to import the data into pandas dataframes. We can use `pd.read_sql_query` or `pd.read_sql_table` or for convenience `pd.read_sql`.

This function is a convenience wrapper around read_sql_table and read_sql_query (for backward compatibility). It will delegate to the specific function depending on the provided input. A SQL query will be routed to read_sql_query , while a database table name will be routed to read_sql_table . Note that the delegated function might have more specific notes about their functionality not listed here.

In [ ]:
# Open connection again because we closed it
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

In [ ]:
# import the data into a pandas dataframe
query_string = """
SELECT *
FROM eda.king_county_house_sales AS sales
JOIN eda.king_county_house_details AS details
  ON sales.house_id = details.id
LIMIT 10
"""
df_psycopg = pd.read_sql(query_string, conn)

In [ ]:
# close the connection
conn.close()

In [ ]:
df_psycopg.head()

In [ ]:
# export the data to a csv-file
df_psycopg.to_csv("data/eda.csv", index=False)
# psycopg2 section is just a connection demo — the real export happens
# further down via the sqlalchemy engine, so no need to write a file here

### Connecting and retrieving data via SQLAlchemy

`sqlalchemy` works similarly. Here you have to create an engine with the database string (a link that includes every information we entered in the conn object)

In [ ]:
from sqlalchemy import create_engine

# read the database string from the .env
load_dotenv()

DB_STRING = os.getenv("DB_STRING")

if DB_STRING is None:
    raise ValueError("DB_STRING is not set in the environment.")

db = create_engine(DB_STRING)

And then you can import that engine with a query into a pandas dataframe.

In [ ]:
# import the data to a pandas dataframe
query_string = """
SELECT *
FROM eda.king_county_house_sales AS sales
JOIN eda.king_county_house_details AS details
  ON sales.house_id = details.id
"""
df_sqlalchemy = pd.read_sql(query_string, db)

Before trusting this join, verify it directly against the database: check for orphans in both directions (a `house_id` in `sales` with no matching `id` in `details`, and vice versa), and confirm which side of the join is the "many" side by counting properties sold more than once.

In [ ]:
# Orphan check, both directions, using the same sqlalchemy engine created above.
# A LEFT JOIN followed by "WHERE <right side> IS NULL" surfaces rows on the
# left that have no match on the right.
orphan_sales_query = """
SELECT COUNT(*) AS orphan_sales
FROM eda.king_county_house_sales AS sales
LEFT JOIN eda.king_county_house_details AS details
  ON sales.house_id = details.id
WHERE details.id IS NULL
"""

orphan_details_query = """
SELECT COUNT(*) AS orphan_details
FROM eda.king_county_house_details AS details
LEFT JOIN eda.king_county_house_sales AS sales
  ON details.id = sales.house_id
WHERE sales.house_id IS NULL
"""

print("Orphan sales (house_id with no matching details.id):")
print(pd.read_sql(orphan_sales_query, db))

print("\nOrphan details (id never referenced by any sale):")
print(pd.read_sql(orphan_details_query, db))

In [ ]:
# Confirm which side of the join is the "many" side: count properties
# that were sold more than once in the sales window.
repeat_sales_query = """
SELECT COUNT(*) AS repeat_sale_properties
FROM (
    SELECT house_id
    FROM eda.king_county_house_sales
    GROUP BY house_id
    HAVING COUNT(*) > 1
) AS repeats
"""

print(pd.read_sql(repeat_sales_query, db))

Both orphan counts should be 0, confirming a clean 1-to-1 match between every `house_id` in `sales` and every `id` in `details`. The repeat-sale count (176) confirms `sales` is the grain-defining ("many") side of the join — those are the properties that were sold more than once in the May 2014–May 2015 window.

In [ ]:
df_sqlalchemy.head()

Because we don't want to run the queries over and over again we can export the data into a .csv file in order to use it in other notebooks as well. 

In [ ]:
# export the data to a csv-file
df_sqlalchemy.to_csv("data/eda.csv", index=False)

In [ ]:
# import the data from a csv-file
df_import = pd.read_csv("data/eda.csv")